# Day 068 — Solution: Document Reader

In [ ]:
_READER_SRC = '"""document_reader.py — Day 068: OCR & Document AI.\n\nExtracts text from images (via Tesseract OCR) and PDFs (via pypdf).\nAlso parses numeric values from OCR output using regex.\n\nSetup:\n    brew install tesseract        # macOS — installs the OCR engine\n    pip install pytesseract pypdf  # Python bindings\n\nUsage:\n    from document_reader import DocumentReader\n    from PIL import Image\n\n    dr = DocumentReader()\n\n    # Read an image\n    img = Image.open("receipt.png")\n    result = dr.read_image(img)\n    print(result["text"])\n    print(result["numbers"])\n\n    # Read a PDF\n    with open("invoice.pdf", "rb") as f:\n        result = dr.read_pdf(f.read())\n    for page in result["pages"]:\n        print(page["text"])\n\nTesting without Tesseract:\n    mock = lambda img: "Hello World\\n$12.99"\n    dr = DocumentReader(ocr_fn=mock)\n"""\nimport re\nimport io\nfrom PIL import Image, ImageEnhance, ImageFilter\n\n\ndef preprocess_for_ocr(img: Image.Image) -> Image.Image:\n    """Improve image quality for OCR.\n\n    Converts to grayscale, boosts contrast, and upscales small images.\n    These steps significantly improve Tesseract accuracy on low-res or\n    low-contrast scans.\n    """\n    out = img.convert("L")\n    out = ImageEnhance.Contrast(out).enhance(2.0)\n    w, h = out.size\n    if w < 1000:\n        scale = 1000 / w\n        out = out.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)\n    return out\n\n\ndef ocr_image(img: Image.Image, ocr_fn=None,\n              lang: str = "eng", config: str = "") -> str:\n    """Extract text from a PIL Image using Tesseract OCR.\n\n    Args:\n        img:    PIL Image to OCR\n        ocr_fn: callable(img) -> str for testing (bypasses Tesseract)\n        lang:   Tesseract language code (default "eng")\n        config: extra Tesseract config flags (e.g. "--psm 6")\n    Returns:\n        Extracted text string (may be empty)\n    """\n    if ocr_fn is not None:\n        return ocr_fn(img)\n    import pytesseract\n    return pytesseract.image_to_string(img, lang=lang, config=config)\n\n\ndef extract_numbers(text: str) -> list:\n    """Parse all numeric values from a text string.\n\n    Handles integers, decimals, and currency amounts.\n    Removes commas from numbers like "1,234.56" before parsing.\n\n    Returns:\n        Sorted list of float values found in text\n    """\n    pattern = r"\\b\\d{1,3}(?:,\\d{3})*(?:\\.\\d+)?|\\b\\d+(?:\\.\\d+)?\\b"\n    raw = re.findall(pattern, text)\n    result = []\n    for s in raw:\n        try:\n            result.append(float(s.replace(",", "")))\n        except ValueError:\n            pass\n    return sorted(result)\n\n\ndef extract_pdf_text(pdf_bytes: bytes) -> list:\n    """Extract text from each page of a PDF.\n\n    Uses pypdf for text-based PDFs. Returns an empty string for image-only\n    (scanned) pages — use ocr_image on those pages instead.\n\n    Args:\n        pdf_bytes: raw PDF bytes (e.g. from open("f.pdf","rb").read())\n    Returns:\n        List of dicts: [{page: int (1-based), text: str, chars: int}]\n    """\n    import pypdf\n    reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))\n    pages = []\n    for i, page in enumerate(reader.pages, start=1):\n        text = page.extract_text() or ""\n        pages.append({"page": i, "text": text, "chars": len(text)})\n    return pages\n\n\nclass DocumentReader:\n    """OCR and PDF text extractor.\n\n    Pass ocr_fn for testing without a Tesseract installation::\n\n        mock = lambda img: "Extracted text"\n        dr = DocumentReader(ocr_fn=mock)\n    """\n\n    def __init__(self, ocr_fn=None) -> None:\n        self._ocr_fn = ocr_fn\n\n    def read_image(self, img: Image.Image,\n                   preprocess: bool = True) -> dict:\n        """OCR a PIL Image and return structured result.\n\n        Args:\n            img:        PIL Image to read\n            preprocess: if True, apply grayscale + contrast + scale first\n        Returns:\n            {text: str, numbers: list[float], chars: int, preprocess: bool}\n        """\n        work = preprocess_for_ocr(img) if preprocess else img\n        text = ocr_image(work, ocr_fn=self._ocr_fn)\n        return {\n            "text":       text,\n            "numbers":    extract_numbers(text),\n            "chars":      len(text),\n            "preprocess": preprocess,\n        }\n\n    def read_pdf(self, pdf_bytes: bytes) -> dict:\n        """Extract text from all pages of a PDF.\n\n        Returns:\n            {pages: list[{page, text, chars}], total_chars: int, page_count: int}\n        """\n        pages = extract_pdf_text(pdf_bytes)\n        return {\n            "pages":       pages,\n            "total_chars": sum(p["chars"] for p in pages),\n            "page_count":  len(pages),\n        }\n\n    def read(self, source, source_type: str = "image") -> dict:\n        """Dispatch to read_image or read_pdf based on source_type.\n\n        Args:\n            source:      PIL Image for "image"; bytes for "pdf"\n            source_type: "image" or "pdf"\n        Returns:\n            Same dict as read_image or read_pdf\n        Raises:\n            ValueError for unknown source_type\n        """\n        if source_type == "image":\n            return self.read_image(source)\n        if source_type == "pdf":\n            return self.read_pdf(source)\n        raise ValueError(\n            f"Unknown source_type: {source_type!r}. Use \'image\' or \'pdf\'."\n        )\n'
from pathlib import Path
Path('document_reader.py').write_text(_READER_SRC, encoding='utf-8')
print('document_reader.py written.')

In [ ]:
import io
import re
from PIL import Image, ImageDraw
from document_reader import DocumentReader, preprocess_for_ocr, extract_numbers, extract_pdf_text

# Mock OCR — no Tesseract required
_mock_ocr = lambda img: 'Coffee $3.50\nSandwich $7.25\nTOTAL $10.75'

dr = DocumentReader(ocr_fn=_mock_ocr)
img = Image.new('RGB', (300, 100), 'white')

# 1. preprocess_for_ocr produces grayscale
preprocessed = preprocess_for_ocr(img)
assert preprocessed.mode == 'L'
print("\u2705 preprocess_for_ocr returns L-mode image")

# 2. read_image returns text/numbers/chars
result = dr.read_image(img)
assert 'text' in result and 'numbers' in result and 'chars' in result
print("\u2705 read_image returns {text, numbers, chars}")

# 3. numbers extracted
assert 3.5 in result['numbers']
assert 10.75 in result['numbers']
print(f"\u2705 numbers: {result['numbers']}")

# 4. extract_numbers standalone
nums = extract_numbers('Price $1,234.56 and $99.99')
assert 1234.56 in nums and 99.99 in nums
print(f"\u2705 extract_numbers handles comma-thousands: {nums}")

# 5. read dispatches correctly
result2 = dr.read(img, source_type='image')
assert result2['text'] == result['text']
print("\u2705 dr.read() dispatches to read_image for source_type='image'")

# 6. unknown source_type raises ValueError
raised = False
try:
    dr.read(img, source_type='video')
except ValueError:
    raised = True
assert raised
print("\u2705 unknown source_type raises ValueError")

# 7. read_pdf from in-memory PDF
def _minimal_pdf_bytes():
    # Tiny valid single-page PDF
    stream = b'BT /F1 12 Tf 50 750 Td (Hello PDF) Tj ET'
    slen = len(stream)
    objs = {
        1: b'<< /Type /Catalog /Pages 2 0 R >>',
        2: b'<< /Type /Pages /Kids [3 0 R] /Count 1 >>',
        3: b'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 << /Type /Font /Subtype /Type1 /BaseFont /Helvetica >> >> >> /Contents 4 0 R >>',
        4: f'<< /Length {slen} >>\nstream\n'.encode() + stream + b'\nendstream',
    }
    buf = io.BytesIO()
    buf.write(b'%PDF-1.4\n')
    offs = {}
    for num, ob in objs.items():
        offs[num] = buf.tell()
        buf.write(f'{num} 0 obj\n'.encode()); buf.write(ob); buf.write(b'\nendobj\n')
    xp = buf.tell()
    buf.write(b'xref\n'); buf.write(f'0 {len(objs)+1}\n'.encode())
    buf.write(b'0000000000 65535 f \n')
    for n in range(1, len(objs)+1):
        buf.write(f'{offs[n]:010d} 00000 n \n'.encode())
    buf.write(f'trailer << /Size {len(objs)+1} /Root 1 0 R >>\nstartxref\n{xp}\n%%EOF\n'.encode())
    return buf.getvalue()

pdf_bytes = _minimal_pdf_bytes()
pdf_result = dr.read_pdf(pdf_bytes)
assert pdf_result['page_count'] == 1
assert isinstance(pdf_result['pages'], list)
print("\u2705 read_pdf extracts 1 page from in-memory PDF")

# 8. extract_pdf_text page numbering
pages = extract_pdf_text(pdf_bytes)
assert pages[0]['page'] == 1
print("\u2705 extract_pdf_text uses 1-based page numbering")

print("\nOCR & Document AI complete!")
